# UROP MATR Anomaly Detection - Local Version

이 노트북은 로컬 폴더에 있는 프로젝트 코드와 MATR 데이터셋을 바로 사용한다.

기본 경로:

```text
PROJECT_DIR = 현재 작업 폴더, 또는 C:/Users/kyucho/UROP 자동 탐색
MATR_DIR    = PROJECT_DIR/MATR
```

다른 위치를 쓰려면 첫 번째 코드 셀을 실행하기 전에 환경변수 `PROJECT_DIR`, `MATR_DIR`를 지정하면 된다.


## 1. 로컬 경로 설정


In [ ]:
from pathlib import Path
import os
import sys
import json
import importlib
import subprocess

def resolve_project_dir():
    candidates = []

    env_project = os.environ.get('PROJECT_DIR')
    if env_project:
        candidates.append(Path(env_project))

    # Prefer a complete project near the running kernel, then common local/Colab paths.
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        cwd / 'UROP',
        Path.home() / 'UROP',
        Path(r'C:/Users/kyucho/UROP'),
        Path('/content/UROP'),
        Path('/content/drive/MyDrive/UROP'),
    ])

    required_script_marker = '--write-validation-predictions'
    required_project_files = [
        Path('scripts/run_matr_locked_test_evaluation.py'),
        Path('scripts/matr_anomaly_scoring.py'),
        Path('scripts/matr_curve_pattern_analysis.py'),
        Path('scripts/matr_functional_pattern_analysis.py'),
        Path('scripts/run_matr_oof_cross_validation.py'),
        Path('scripts/matr_oof_residual_analysis.py'),
    ]
    unique_candidates = []
    seen_candidates = set()
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if resolved not in seen_candidates:
            unique_candidates.append(resolved)
            seen_candidates.add(resolved)

    audit_lines = []
    for candidate in unique_candidates:
        missing = [
            str(relative_path)
            for relative_path in required_project_files
            if not (candidate / relative_path).is_file()
        ]
        if missing:
            audit_lines.append(f'{candidate}: missing {", ".join(missing)}')
            continue
        run_script = candidate / required_project_files[0]
        script_text = run_script.read_text(encoding='utf-8', errors='ignore')
        if required_script_marker in script_text:
            return candidate
        audit_lines.append(
            f'{candidate}: evaluation script lacks {required_script_marker}'
        )

    checked = '\n'.join(audit_lines)
    expected_scripts_dir = cwd / 'scripts'
    raise FileNotFoundError(
        'Could not find a complete updated UROP project directory. Checked:\n'
        + checked
        + '\n\nKeep the notebook and the complete scripts folder together. '
        + 'Expected scripts folder:\n'
        + str(expected_scripts_dir)
        + '\nYou may also set the PROJECT_DIR environment variable explicitly.'
    )

PROJECT_DIR = resolve_project_dir()
WORK_DIR = Path.cwd().resolve()
MATR_DIR = Path(os.environ.get('MATR_DIR', PROJECT_DIR / 'MATR')).expanduser().resolve()
SCRIPTS_DIR = (PROJECT_DIR / 'scripts').resolve()
project_path = str(PROJECT_DIR)
sys.path[:] = [entry for entry in sys.path if entry != project_path]
sys.path.insert(0, project_path)
importlib.invalidate_caches()

# Remove only a stale/shadowing `scripts` package left in this kernel.
loaded_scripts = sys.modules.get('scripts')
if loaded_scripts is not None:
    loaded_locations = []
    package_file = getattr(loaded_scripts, '__file__', None)
    if package_file:
        loaded_locations.append(Path(package_file).resolve().parent)
    package_path = getattr(loaded_scripts, '__path__', [])
    loaded_locations.extend(Path(path).resolve() for path in package_path)
    if SCRIPTS_DIR not in loaded_locations:
        stale_modules = [
            name for name in list(sys.modules)
            if name == 'scripts' or name.startswith('scripts.')
        ]
        for name in stale_modules:
            sys.modules.pop(name, None)
        importlib.invalidate_caches()

print('PROJECT_DIR:', PROJECT_DIR)
print('WORK_DIR:', WORK_DIR)
print('MATR_DIR:', MATR_DIR)
print('FUNCTIONAL_MODULE:', SCRIPTS_DIR / 'matr_functional_pattern_analysis.py')
print('PROJECT_DIR exists:', PROJECT_DIR.exists())
print('MATR_DIR exists:', MATR_DIR.exists())


## 2. 로컬 파일 및 설정 확인


In [ ]:
# Current SOH configuration:
# - battery-level split
# - sliding-window samples
# - lookback=10
# - scoring horizons=10,50,100 (alpha selection remains H50/H100)
# - H10 reuses the locked configuration/checkpoint but did not participate
#   in the original H50/H100 Optuna hyperparameter-selection objective.
# - target_scale=100, fixed_len=100
CONFIG_DIR = PROJECT_DIR / 'outputs' / 'matr_step7_sliding_l10_optuna_cpdsconv_h50_h100_wide'
CONFIG_PATH = CONFIG_DIR / 'best_optuna_config.json'
TUNING_CONTEXT_PATH = CONFIG_DIR / 'optuna_tuning_config.json'
LOCKED_TEST_DIR = PROJECT_DIR / 'outputs' / 'matr_step7_locked_test_cpdsconv_l10_h10_h50_h100_wide_best'
CHECKPOINT_ROOT = LOCKED_TEST_DIR / 'checkpoints'
SPLIT_MANIFEST_ROOT = LOCKED_TEST_DIR
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'matr_locked_test_sliding_l10_h10_h50_h100_wide_best_anomaly_inference'
RUN_SCRIPT = PROJECT_DIR / 'scripts' / 'run_matr_locked_test_evaluation.py'
OOF_RUN_SCRIPT = PROJECT_DIR / 'scripts' / 'run_matr_oof_cross_validation.py'
OOF_OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'matr_oof_crossfit_l10_h10_h50_h100_r3_k5_v2'

# MATR data is expected to be an already-extracted local folder.
pkl_files = sorted(MATR_DIR.rglob('*.pkl')) if MATR_DIR.exists() else []

print('PROJECT_DIR:', PROJECT_DIR)
print('MATR_DIR:', MATR_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('RUN_SCRIPT exists:', RUN_SCRIPT.exists(), RUN_SCRIPT)
print('OOF_RUN_SCRIPT exists:', OOF_RUN_SCRIPT.exists(), OOF_RUN_SCRIPT)
print('OOF_OUTPUT_DIR:', OOF_OUTPUT_DIR)
print('CONFIG_PATH exists:', CONFIG_PATH.exists(), CONFIG_PATH)
print('TUNING_CONTEXT_PATH exists:', TUNING_CONTEXT_PATH.exists(), TUNING_CONTEXT_PATH)
print('CHECKPOINT_ROOT exists:', CHECKPOINT_ROOT.exists(), CHECKPOINT_ROOT)
print('SPLIT_MANIFEST_ROOT exists:', SPLIT_MANIFEST_ROOT.exists(), SPLIT_MANIFEST_ROOT)
print('PKL file count:', len(pkl_files))
for path in pkl_files[:10]:
    print(path)

if not RUN_SCRIPT.exists():
    raise FileNotFoundError(f'Missing run script: {RUN_SCRIPT}')
if not OOF_RUN_SCRIPT.exists():
    raise FileNotFoundError(f'Missing OOF cross-validation script: {OOF_RUN_SCRIPT}')
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        'Missing tuned config file. Put '
        'outputs/matr_step7_sliding_l10_optuna_cpdsconv_h50_h100_wide/best_optuna_config.json '
        'inside the uploaded project folder.'
    )
if not TUNING_CONTEXT_PATH.exists():
    raise FileNotFoundError(
        'Missing Optuna sidecar config. Put '
        'outputs/matr_step7_sliding_l10_optuna_cpdsconv_h50_h100_wide/optuna_tuning_config.json '
        'next to best_optuna_config.json.'
    )
if not CHECKPOINT_ROOT.is_dir():
    raise FileNotFoundError(f'Missing locked checkpoint directory: {CHECKPOINT_ROOT}')
for seed in [42, 43, 44]:
    manifest_path = SPLIT_MANIFEST_ROOT / f'split_manifest_seed{seed}.json'
    if not manifest_path.exists():
        raise FileNotFoundError(f'Missing original split manifest: {manifest_path}')
if not pkl_files:
    raise RuntimeError(
        f'No .pkl files found under MATR_DIR={MATR_DIR}. '
        'Place the already-extracted MATR dataset folder there, or set os.environ["MATR_DIR"] before running setup.'
    )


## 3. Anomaly score 정의와 α 선택 방법

이 노트북의 열화 이상 점수는 다음과 같이 정의한다.

```text
score = α × degradation_residual_z + (1 - α) × degradation_slope_z
```

실제 이상 정답 라벨이 없으므로 α를 AP/ROC-AUC로 최적화하지 않는다. 전체 169개 셀을 배터리 단위 5-fold로 나누고 이를 3회 반복하여, 각 셀이 학습·정규화·early stopping에서 제외된 상태의 OOF 예측 잔차를 만든다. 각 fold의 inner-validation 셀에서 component scale, α, horizon scale을 정한 뒤 H10/H50/H100 공통 window의 평균 severity와 세 horizon 중앙값으로 셀 점수를 계산한다. 이 inner-validation은 early stopping 뒤 점수 변환에도 재사용되므로 독립 calibration이라고 부르지 않는다. 점수는 주로 `실제 열화가 예측보다 빠른 경우`를 보는 단방향 보조지표다. 최종 희귀도는 같은 배치에서 관측 길이가 가까운 OOF 셀들과 비교하며, coverage 연관성과 fold별 점수 이동이 크면 후보 판정을 보류한다. 3회 중 2회 이상에서 같은 tail evidence가 있을 때만 안정적인 잔차 후보로 표시한다. 이 값은 고장 확률이나 정답 p-value가 아니라 cross-fitted empirical tail rank이며, 함수 기반 전체 SOH 곡선 분석과 별도로 비교한다.


In [ ]:
import numpy as np
import pandas as pd

from scripts.matr_anomaly_scoring import (
    AnomalyConfig,
    add_scores_by_seed,
    aggregate_cell_nonconformity,
    aggregate_physical_cell_evidence,
    audit_cell_score_coverage,
    apply_component_calibration,
    apply_empirical_conformal_pvalues,
    apply_horizon_severity_calibration,
    assert_seed_split_isolation,
    fit_component_calibration,
    fit_horizon_severity_calibration,
    plot_alpha_search,
    prepare_residual_features,
    select_alpha_by_seed,
    split_validation_roles,
    summarize_common_horizon_scores,
)
from scripts.matr_oof_residual_analysis import (
    OOFResidualConfig,
    analyze_oof_residual_artifacts,
    compare_functional_and_oof_evidence,
)

ANOMALY_CONFIG = AnomalyConfig(
    target_model='cpmlp_cpdsconv_fusion',
    expected_horizons=(10, 50, 100),
    alpha_selection_horizons=(50, 100),
    alpha_grid=tuple(float(x) for x in np.round(np.linspace(0.0, 1.0, 21), 2)),
    validation_selection_fraction=1.0 / 3.0,
    alpha_cell_aggregation='mean',
    min_common_windows=5,
    min_paired_cells=5,
    bootstrap_repeats=500,  # 최종 보고에서는 1000 이상 권장
    conformal_candidate_p=0.10,
    conformal_strong_p=0.05,
    random_state=20260711,
    threshold_method='cell_empirical_conformal',
)
OOF_RESIDUAL_CONFIG = OOFResidualConfig(
    target_model='cpmlp_cpdsconv_fusion',
    expected_horizons=(10, 50, 100),
    alpha_selection_horizons=(50, 100),
    alpha_grid=tuple(float(x) for x in np.round(np.linspace(0.0, 1.0, 21), 2)),
    min_common_windows=5,
    min_paired_cells=5,
    bootstrap_repeats=500,
    candidate_p=0.10,
    strong_p=0.05,
    coverage_reference_cells=20,
    minimum_coverage_reference_cells=9,
    coverage_warning_abs_rank_corr=0.50,
    required_repeat_fraction=2.0 / 3.0,
    random_state=20260715,
    fold_warning_adjusted_rank_effect=0.10,
    fold_warning_permutation_p=0.05,
    fold_permutation_repeats=1000,
    minimum_fold_cells=3,
)

print('Anomaly score: alpha * residual_z + (1 - alpha) * slope_z')
print('Alpha grid:', OOF_RESIDUAL_CONFIG.alpha_grid)
print('Scoring horizons:', OOF_RESIDUAL_CONFIG.expected_horizons)
print('Alpha selection horizons:', OOF_RESIDUAL_CONFIG.alpha_selection_horizons)
print('Threshold method: repeated OOF same-batch coverage-matched empirical tail rank')
print('Cell score: median of H10/H50/H100 relative mean severities')
print('Candidate tail rank <=', OOF_RESIDUAL_CONFIG.candidate_p)
print('Strong tail rank <=', OOF_RESIDUAL_CONFIG.strong_p)
print('Stable evidence requires at least 2 of 3 OOF repeats.')
print('Coverage and outer-fold score-shift audits can veto residual candidates.')


## 4. GPU 확인


In [ ]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('DEVICE:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 5. 최신 코드 실행


In [ ]:
help_result = subprocess.run(
    [sys.executable, str(RUN_SCRIPT), '--help'],
    cwd=PROJECT_DIR,
    text=True,
    capture_output=True,
)

help_text = help_result.stdout + '\n' + help_result.stderr
print('Python executable:', sys.executable)
print('Evaluation script:', RUN_SCRIPT.resolve())
print('Help return code:', help_result.returncode)
if help_result.returncode != 0:
    raise RuntimeError(
        'The evaluation script could not start, so its options could not be checked. '
        'This usually means the notebook kernel is using the wrong Python environment '
        'or a required package is missing.\n'
        + f'Python: {sys.executable}\n'
        + f'Script: {RUN_SCRIPT.resolve()}\n\nSTDERR:\n'
        + help_result.stderr[-4000:]
    )
required_options = [
    '--sample-mode', '--lookback', '--horizons', '--include-references',
    '--inference-only', '--checkpoint-root', '--split-manifest-root',
    '--write-validation-predictions',
]
missing_options = [opt for opt in required_options if opt not in help_text]
if missing_options:
    raise RuntimeError(
        'The evaluation script is not the updated version. Missing options: '
        + ', '.join(missing_options)
        + f'\nScript: {RUN_SCRIPT.resolve()}'
        + f'\nPython: {sys.executable}'
        + '\n\nSTDERR:\n'
        + help_result.stderr[-4000:]
    )

config_rel = CONFIG_PATH.relative_to(PROJECT_DIR).as_posix()
output_rel = OUTPUT_DIR.relative_to(PROJECT_DIR).as_posix()
checkpoint_root_rel = CHECKPOINT_ROOT.relative_to(PROJECT_DIR).as_posix()
split_manifest_root_rel = SPLIT_MANIFEST_ROOT.relative_to(PROJECT_DIR).as_posix()

cmd = [
    sys.executable,
    str(RUN_SCRIPT),
    '--data-root', str(MATR_DIR),
    '--config-path', config_rel,
    '--output-dir', output_rel,
    '--checkpoint-root', checkpoint_root_rel,
    '--split-manifest-root', split_manifest_root_rel,
    '--inference-only',
    '--write-validation-predictions',
    '--device', DEVICE,
    '--include-references',
    '--sample-mode', 'sliding-window',
    '--lookback', '10',
    '--horizons', '10', '50', '100',
    '--seeds', '42', '43', '44',
    '--target-scale', '100',
    '--fixed-len', '100',
]

print('Running command:')
print(' '.join(cmd))

result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
print('return code:', result.returncode)
print('\n--- STDOUT tail ---')
print(result.stdout[-8000:])
print('\n--- STDERR tail ---')
print(result.stderr[-8000:])

if result.returncode != 0:
    raise RuntimeError('Model evaluation command failed. Check STDERR above.')


## 6. 필수 결과 파일 확인


In [ ]:
import json

required_outputs = [
    OUTPUT_DIR / 'locked_test_summary.csv',
    OUTPUT_DIR / 'test_predictions.csv',
    OUTPUT_DIR / 'validation_predictions.csv',
    OUTPUT_DIR / 'test_summary_by_model_horizon.csv',
    OUTPUT_DIR / 'locked_test_config.json',
]

for path in required_outputs:
    print(path.name, path.exists(), path)

missing = [path for path in required_outputs if not path.exists()]
if missing:
    print('OUTPUT_DIR listing:')
    if OUTPUT_DIR.exists():
        for p in sorted(OUTPUT_DIR.rglob('*'))[:200]:
            print(p.relative_to(OUTPUT_DIR), 'dir' if p.is_dir() else p.stat().st_size)
    raise FileNotFoundError('Missing result files: ' + ', '.join(str(p) for p in missing))

with open(OUTPUT_DIR / 'locked_test_config.json', 'r', encoding='utf-8') as f:
    locked_test_config = json.load(f)

runtime_config = locked_test_config.get('runtime_config', {})
print('Runtime config:', runtime_config)
print('Execution mode:', locked_test_config.get('execution_mode'))

expected_runtime = {
    'lookback': 10,
    'sample_mode': 'sliding-window',
    'horizons': [10, 50, 100],
    'seeds': [42, 43, 44],
    'target_scale': 100.0,
    'fixed_len': 100,
}

expected_audit = {
    'execution_mode': 'checkpoint_inference_only',
    'training_performed': False,
    'validation_selection_performed': False,
    'hyperparameter_tuning_performed': False,
    'batch_specific_fine_tuning_performed': False,
    'validation_predictions_written': True,
}

for key, expected in expected_audit.items():
    actual = locked_test_config.get(key)
    if actual != expected:
        raise RuntimeError(f'Unexpected locked_test_config[{key!r}]: expected {expected!r}, got {actual!r}')

for key, expected in expected_runtime.items():
    actual = runtime_config.get(key)
    if actual != expected:
        raise RuntimeError(f'Unexpected runtime_config[{key!r}]: expected {expected!r}, got {actual!r}')

for seed in expected_runtime['seeds']:
    split_path = OUTPUT_DIR / f'split_manifest_seed{seed}.json'
    with open(split_path, 'r', encoding='utf-8') as f:
        split_manifest = json.load(f)
    if split_manifest.get('reused_without_resplitting') is not True:
        raise RuntimeError(f'Original split was not reused: {split_path}')

print('Inference-only audit passed: no training/tuning and original battery splits were reused.')
print('Required model result files exist and runtime config matches the intended sliding-window l10 h10/h50/h100 setup.')


## 6-1. 전체 169개 셀 반복 OOF 교차검증

기존 checkpoint는 학습에 포함된 셀까지 보았으므로 전체 셀의 잔차 비교에 재사용하지 않는다. 배터리 단위 5-fold를 3회 반복하고, 각 outer fold마다 모델과 train-only 정규화를 새로 학습한다. 모델 종류와 hyperparameter는 기존 선택값으로 고정하므로 결과는 완전한 nested CV가 아니라 `cross-fitted diagnostic`이다.


In [ ]:
OOF_REPEAT_SEEDS = [42, 43, 44]
OOF_N_SPLITS = 5
OOF_COMPLETION_PATH = OOF_OUTPUT_DIR / 'oof_completion.json'
OOF_RUN_CONFIG_PATH = OOF_OUTPUT_DIR / 'oof_run_config.json'
OOF_REQUIRED_OUTPUTS = [
    OOF_OUTPUT_DIR / 'oof_predictions.csv',
    OOF_OUTPUT_DIR / 'inner_validation_predictions.csv',
    OOF_OUTPUT_DIR / 'oof_fold_assignments.csv',
    OOF_OUTPUT_DIR / 'oof_split_roles.csv',
    OOF_OUTPUT_DIR / 'oof_metrics_summary.csv',
    OOF_RUN_CONFIG_PATH,
    OOF_COMPLETION_PATH,
]

OOF_REQUIRED_PREDICTION_COLUMNS = {
    'run_id', 'stage', 'repeat_seed', 'outer_fold',
    'normalizer_fit_role', 'split',
}
oof_artifacts_are_current = OOF_COMPLETION_PATH.is_file()
if oof_artifacts_are_current:
    for prediction_name in ['oof_predictions.csv', 'inner_validation_predictions.csv']:
        prediction_path = OOF_OUTPUT_DIR / prediction_name
        if not prediction_path.is_file():
            oof_artifacts_are_current = False
            break
        prediction_columns = set(pd.read_csv(prediction_path, nrows=0).columns)
        if not OOF_REQUIRED_PREDICTION_COLUMNS.issubset(prediction_columns):
            oof_artifacts_are_current = False
            break

if not oof_artifacts_are_current:
    oof_cmd = [
        sys.executable, str(OOF_RUN_SCRIPT),
        '--data-root', str(MATR_DIR),
        '--config-path', str(CONFIG_PATH),
        '--output-dir', str(OOF_OUTPUT_DIR),
        '--n-splits', str(OOF_N_SPLITS),
        '--repeat-seeds', *[str(seed) for seed in OOF_REPEAT_SEEDS],
        '--inner-validation-fraction', '0.20',
        '--expected-battery-count', '169',
        '--device', DEVICE,
        '--sample-mode', 'sliding-window',
        '--lookback', '10',
        '--horizons', '10', '50', '100',
        '--target-scale', '100',
        '--fixed-len', '100',
    ]
    if OOF_OUTPUT_DIR.exists() and any(OOF_OUTPUT_DIR.iterdir()):
        oof_cmd.append('--resume')
    print('Running full-cohort OOF command (45 model fits):')
    print(' '.join(oof_cmd))
    oof_process = subprocess.run(oof_cmd, cwd=PROJECT_DIR)
    if oof_process.returncode != 0:
        raise RuntimeError('OOF cross-validation failed. Re-run this cell to resume completed folds.')
else:
    print('Reusing completed OOF run:', OOF_OUTPUT_DIR)

missing_oof = [path for path in OOF_REQUIRED_OUTPUTS if not path.is_file()]
if missing_oof:
    raise FileNotFoundError('Missing OOF outputs: ' + ', '.join(str(path) for path in missing_oof))
with open(OOF_COMPLETION_PATH, 'r', encoding='utf-8') as handle:
    oof_completion = json.load(handle)
with open(OOF_RUN_CONFIG_PATH, 'r', encoding='utf-8') as handle:
    oof_run_config = json.load(handle)
if oof_completion.get('status') != 'complete':
    raise RuntimeError(f'OOF run is not complete: {oof_completion}')
if oof_completion.get('oof_unique_batteries') != 169:
    raise RuntimeError(f"Expected 169 OOF batteries, got {oof_completion.get('oof_unique_batteries')}")
if oof_completion.get('repeat_seeds') != OOF_REPEAT_SEEDS:
    raise RuntimeError('OOF repeat seeds do not match the analysis plan')
if oof_completion.get('n_splits') != OOF_N_SPLITS:
    raise RuntimeError('OOF fold count does not match the analysis plan')
if oof_completion.get('horizons') != [10, 50, 100]:
    raise RuntimeError('OOF horizons do not match H10/H50/H100')
if oof_completion.get('selected_model') != OOF_RESIDUAL_CONFIG.target_model:
    raise RuntimeError('OOF selected model does not match residual analysis')
expected_horizon_coverage = {'10': 169, '50': 169, '100': 169}
if oof_completion.get('eligible_batteries_by_horizon') != expected_horizon_coverage:
    raise RuntimeError(
        'OOF per-horizon cohort mismatch: ' +
        repr(oof_completion.get('eligible_batteries_by_horizon'))
    )
resolved_oof = oof_run_config.get('resolved_runtime_config', {})
expected_oof_runtime = {
    'lookback': 10, 'sample_mode': 'sliding-window',
    'horizons': [10, 50, 100], 'target_scale': 100.0, 'fixed_len': 100,
}
for key, expected in expected_oof_runtime.items():
    if resolved_oof.get(key) != expected:
        raise RuntimeError(
            f'OOF runtime mismatch for {key}: ' 
            f'{resolved_oof.get(key)!r} != {expected!r}'
        )
print('OOF coverage audit passed:', oof_completion)


## 7. 기존 모델 vs 개선 모델 성능 비교


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

ANALYSIS_RUN_ID = globals().get('ANALYSIS_RUN_ID') or os.environ.get('ANALYSIS_RUN_ID') or datetime.now().strftime('%Y%m%d_%H%M%S')
REPORT_BASE_DIR = OOF_OUTPUT_DIR / 'functional_oof_comparison_runs'
REPORT_DIR = REPORT_BASE_DIR / ANALYSIS_RUN_ID
FIGURE_DIR = REPORT_DIR / 'figures'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

summary_df = pd.read_csv(OUTPUT_DIR / 'locked_test_summary.csv')
horizon_df = pd.read_csv(OUTPUT_DIR / 'test_summary_by_model_horizon.csv')
pred = pd.read_csv(OUTPUT_DIR / 'test_predictions.csv')
val_pred = pd.read_csv(OUTPUT_DIR / 'validation_predictions.csv')

model_order = ['persistence', 'log_degradation', 'cpmlp', 'cpdsconv', 'cpmlp_cpdsconv_fusion']
model_labels = {
    'persistence': 'Persistence',
    'log_degradation': 'Log Degradation',
    'cpmlp': 'CPMLP',
    'cpdsconv': 'CPDSConv',
    'cpmlp_cpdsconv_fusion': 'CPMLP-CPDSConv Fusion',
}
model_colors = {
    'persistence': '#9CA3AF',
    'log_degradation': '#F59E0B',
    'cpmlp': '#10B981',
    'cpdsconv': '#7C3AED',
    'cpmlp_cpdsconv_fusion': '#2563EB',
}

available_order = [m for m in model_order if m in set(summary_df['model'])]
perf = summary_df[summary_df['model'].isin(available_order)].copy()
perf = perf.set_index('model').loc[available_order].reset_index()

display_cols = [c for c in ['model', 'avg_MAE_mean', 'avg_RMSE_mean', 'avg_MAPE_percent_mean', 'average_Skill_MAE_vs_persistence'] if c in perf.columns]
display(perf[display_cols])

if {'cpmlp', 'cpmlp_cpdsconv_fusion'}.issubset(set(summary_df['model'])):
    cpmlp_row = summary_df[summary_df['model'] == 'cpmlp'].iloc[0]
    fusion_row = summary_df[summary_df['model'] == 'cpmlp_cpdsconv_fusion'].iloc[0]
    compare_rows = []
    for col, label in [('avg_MAE_mean', 'MAE'), ('avg_RMSE_mean', 'RMSE'), ('avg_MAPE_percent_mean', 'MAPE')]:
        if col in summary_df.columns:
            compare_rows.append({
                'metric': label,
                'cpmlp': cpmlp_row[col],
                'cpmlp_cpdsconv_fusion': fusion_row[col],
                'delta_percent_vs_cpmlp': (cpmlp_row[col] - fusion_row[col]) / cpmlp_row[col] * 100,
            })
    comparison_df = pd.DataFrame(compare_rows)
    display(comparison_df)
    comparison_df.to_csv(REPORT_DIR / 'fusion_vs_cpmlp_metrics.csv', index=False, encoding='utf-8-sig')

metric_cols = [('avg_MAE_mean', 'MAE'), ('avg_RMSE_mean', 'RMSE'), ('avg_MAPE_percent_mean', 'MAPE (%)')]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (col, label) in zip(axes, metric_cols):
    if col not in perf.columns:
        ax.set_visible(False)
        continue
    bars = ax.bar(
        perf['model'].map(model_labels),
        perf[col],
        color=[model_colors[m] for m in perf['model']],
    )
    ax.set_title(label)
    ax.set_ylabel(label)
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=18)
    for bar in bars:
        h = bar.get_height()
        txt = f'{h:.5f}' if label != 'MAPE (%)' else f'{h:.3f}'
        ax.text(bar.get_x() + bar.get_width()/2, h, txt, ha='center', va='bottom', fontsize=9)
fig.suptitle('Model Performance Comparison: Baselines vs Fusion Model', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURE_DIR / '01_model_performance_comparison.png', dpi=220)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
hmetric_cols = [('MAE_mean', 'MAE'), ('RMSE_mean', 'RMSE'), ('MAPE_percent_mean', 'MAPE (%)')]
for ax, (col, label) in zip(axes, hmetric_cols):
    if col not in horizon_df.columns:
        ax.set_visible(False)
        continue
    for model in available_order:
        sub = horizon_df[horizon_df['model'] == model].sort_values('horizon')
        if sub.empty:
            continue
        ax.plot(sub['horizon'], sub[col], marker='o', linewidth=2, label=model_labels[model], color=model_colors[model])
    ax.set_title(label)
    ax.set_xlabel('Prediction Horizon')
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle('Performance by Horizon', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURE_DIR / '02_performance_by_horizon.png', dpi=220)
plt.show()


## 8. 전체 셀 반복 OOF 예측 잔차 기반 empirical tail-rank 보조분석


In [ ]:
'''
# 1) Build residual components from validation and locked-test predictions.
validation_df = prepare_residual_features(
    val_pred,
    'validation',
    target_model=ANOMALY_CONFIG.target_model,
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
)
df = prepare_residual_features(
    pred,
    'test',
    target_model=ANOMALY_CONFIG.target_model,
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
)
assert_seed_split_isolation(validation_df, df)

# 2) Use about one third of validation cells for score development and keep
#    about two thirds untouched as the empirical conformal reference set.
validation_df, validation_role_table = split_validation_roles(
    validation_df,
    selection_fraction=ANOMALY_CONFIG.validation_selection_fraction,
    random_state=ANOMALY_CONFIG.random_state,
)
alpha_selection_raw = validation_df[
    validation_df['validation_role'] == 'alpha_selection'
].copy()

# 3) Component scaling and alpha selection use alpha-development cells only.
component_calibration = fit_component_calibration(alpha_selection_raw)
validation_df = apply_component_calibration(validation_df, component_calibration)
df = apply_component_calibration(df, component_calibration)
alpha_selection_df = validation_df[
    validation_df['validation_role'] == 'alpha_selection'
].copy()
alpha_by_seed, alpha_search_table = select_alpha_by_seed(
    alpha_selection_df,
    alpha_grid=ANOMALY_CONFIG.alpha_grid,
    horizons=ANOMALY_CONFIG.alpha_selection_horizons,
    severity_aggregation=ANOMALY_CONFIG.alpha_cell_aggregation,
    min_common_windows=ANOMALY_CONFIG.min_common_windows,
    min_paired_cells=ANOMALY_CONFIG.min_paired_cells,
    bootstrap_repeats=ANOMALY_CONFIG.bootstrap_repeats,
    random_state=ANOMALY_CONFIG.random_state,
)
validation_df = add_scores_by_seed(validation_df, alpha_by_seed)
df = add_scores_by_seed(df, alpha_by_seed)

# 4) Convert overlapping windows into one continuous mean severity per
#    cell and horizon, using only input_end_cycle values common to all horizons.
alpha_horizon_summary = summarize_common_horizon_scores(
    validation_df[validation_df['validation_role'] == 'alpha_selection'],
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
    min_common_windows=ANOMALY_CONFIG.min_common_windows,
)
calibration_horizon_summary = summarize_common_horizon_scores(
    validation_df[validation_df['validation_role'] == 'conformal_calibration'],
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
    min_common_windows=ANOMALY_CONFIG.min_common_windows,
)
cell_horizon_summary = summarize_common_horizon_scores(
    df,
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
    min_common_windows=ANOMALY_CONFIG.min_common_windows,
)

# 5) Put H10/H50/H100 cell means on comparable alpha-development scales.
horizon_severity_calibration = fit_horizon_severity_calibration(
    alpha_horizon_summary
)
calibration_horizon_summary = apply_horizon_severity_calibration(
    calibration_horizon_summary, horizon_severity_calibration
)
cell_horizon_summary = apply_horizon_severity_calibration(
    cell_horizon_summary, horizon_severity_calibration
)

# 6) The median of the three relative horizon means is one continuous
#    cell nonconformity score. No window flag, ratio, or binary horizon vote.
calibration_cell_summary = aggregate_cell_nonconformity(
    calibration_horizon_summary,
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
)
unscored_test_cells = aggregate_cell_nonconformity(
    cell_horizon_summary,
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
)
cell_summary, conformal_audit = apply_empirical_conformal_pvalues(
    unscored_test_cells,
    calibration_cell_summary,
    candidate_p=ANOMALY_CONFIG.conformal_candidate_p,
    strong_p=ANOMALY_CONFIG.conformal_strong_p,
)
physical_cell_summary = aggregate_physical_cell_evidence(cell_summary)
coverage_sensitivity_audit = pd.concat(
    [
        audit_cell_score_coverage(
            calibration_cell_summary, data_role='conformal_calibration'
        ),
        audit_cell_score_coverage(cell_summary, data_role='locked_test'),
    ],
    ignore_index=True,
)
batch_candidate_audit = (
    physical_cell_summary.groupby('batch_id', as_index=False)
    .agg(
        evaluated_physical_cells=('battery_id', 'nunique'),
        empirical_tail_candidates=('is_physical_candidate', 'sum'),
        strong_tail_candidates=('is_physical_strong_candidate', 'sum'),
        repeated_seed_candidates=('is_repeated_seed_candidate', 'sum'),
        single_seed_candidates=('is_single_seed_candidate', 'sum'),
        median_worst_seed_p=('worst_seed_empirical_p_value', 'median'),
    )
)
batch_candidate_audit['candidate_rate'] = (
    batch_candidate_audit['empirical_tail_candidates']
    / batch_candidate_audit['evaluated_physical_cells']
)

# 7) Save score provenance, calibration references, and both seed-level and
#    unique physical-cell result tables.
validation_role_table.to_csv(
    REPORT_DIR / 'validation_battery_roles.csv', index=False, encoding='utf-8-sig'
)
alpha_by_seed.to_csv(
    REPORT_DIR / 'selected_anomaly_alpha_by_seed.csv', index=False, encoding='utf-8-sig'
)
alpha_search_table.to_csv(
    REPORT_DIR / 'anomaly_alpha_search_validation.csv', index=False, encoding='utf-8-sig'
)
component_calibration.to_csv(
    REPORT_DIR / 'alpha_development_component_calibration.csv', index=False, encoding='utf-8-sig'
)
horizon_severity_calibration.to_csv(
    REPORT_DIR / 'alpha_development_horizon_calibration.csv', index=False, encoding='utf-8-sig'
)
calibration_cell_summary.to_csv(
    REPORT_DIR / 'validation_cell_conformal_calibration.csv', index=False, encoding='utf-8-sig'
)
conformal_audit.to_csv(
    REPORT_DIR / 'conformal_calibration_audit.csv', index=False, encoding='utf-8-sig'
)
coverage_sensitivity_audit.to_csv(
    REPORT_DIR / 'cell_score_coverage_sensitivity_audit.csv', index=False, encoding='utf-8-sig'
)
df.to_csv(REPORT_DIR / 'cycle_level_degradation_anomaly_scores.csv', index=False, encoding='utf-8-sig')
cell_horizon_summary.to_csv(REPORT_DIR / 'cell_horizon_mean_severity.csv', index=False, encoding='utf-8-sig')
cell_summary.to_csv(REPORT_DIR / 'cell_level_conformal_summary_by_seed.csv', index=False, encoding='utf-8-sig')
physical_cell_summary.to_csv(REPORT_DIR / 'physical_cell_conformal_summary.csv', index=False, encoding='utf-8-sig')
batch_candidate_audit.to_csv(REPORT_DIR / 'physical_cell_conformal_summary_by_batch.csv', index=False, encoding='utf-8-sig')

alpha_figure = plot_alpha_search(
    alpha_search_table,
    alpha_by_seed,
    save_path=FIGURE_DIR / '03_validation_only_alpha_selection.png',
)
plt.show()

print('Stability-selected alpha by seed:')
display(alpha_by_seed)
print('Empirical conformal calibration audit:')
display(conformal_audit)
print('Coverage sensitivity audit (diagnostic only):')
display(coverage_sensitivity_audit)
if coverage_sensitivity_audit['has_large_coverage_association'].any():
    print('WARNING: score rarity is strongly associated with observed life coverage for at least one seed/role; inspect raw SOH curves before interpreting it as abnormal degradation.')
print('Seed-level empirical tail candidates:', int(cell_summary['is_conformal_candidate'].sum()), '/', len(cell_summary))
print('Seed-level strong candidates:', int(cell_summary['is_strong_candidate'].sum()))
print('Unique physical cells evaluated:', len(physical_cell_summary))
print('Physical-cell candidates across all available test seeds:', int(physical_cell_summary['is_physical_candidate'].sum()))
print('Repeated-seed candidates:', int(physical_cell_summary['is_repeated_seed_candidate'].sum()))
print('Single-test-seed candidates:', int(physical_cell_summary['is_single_seed_candidate'].sum()))
print('Batch-level candidate audit (pooled p-values are marginal, not batch-conditional):')
display(batch_candidate_audit)
print('Saved:', REPORT_DIR)
display(physical_cell_summary.head(30))
'''

# Full-cohort repeated OOF residual analysis. The disabled block above is
# retained only as historical documentation of the former 85-cell analysis.
oof_residual_result = analyze_oof_residual_artifacts(
    OOF_OUTPUT_DIR,
    config=OOF_RESIDUAL_CONFIG,
    keep_scored_outer_windows=False,
)
oof_target_parity_rows = oof_residual_result.outer_prediction_parity.copy()
cell_summary = oof_residual_result.cell_score_by_repeat.copy()
physical_cell_summary = oof_residual_result.physical_cell_summary.copy()
component_calibration = oof_residual_result.fold_calibration.copy()
alpha_search_table = oof_residual_result.alpha_search.copy()
coverage_sensitivity_audit = oof_residual_result.coverage_audit.copy()

expected_oof_batteries = int(oof_completion['oof_unique_batteries'])
if len(physical_cell_summary) != expected_oof_batteries:
    raise RuntimeError(
        f'OOF residual summary coverage mismatch: ' 
        f'{len(physical_cell_summary)} != {expected_oof_batteries}'
    )
if not physical_cell_summary['has_complete_oof_residual_evidence'].all():
    incomplete = physical_cell_summary.loc[
        ~physical_cell_summary['has_complete_oof_residual_evidence'], 'battery_id'
    ].tolist()
    print(
        'WARNING: incomplete residual evidence is kept as unavailable, not normal:',
        incomplete[:20],
    )

oof_batch_audit = (
    physical_cell_summary.groupby('batch_id', as_index=False)
    .agg(
        physical_cells=('battery_id', 'nunique'),
        stable_residual_candidates=('is_stable_oof_residual_candidate', 'sum'),
        stable_strong_candidates=(
            'is_stable_oof_residual_strong_candidate', 'sum'
        ),
        confounded_cells=('is_residual_confounding', 'sum'),
        median_rarity_percentile=('median_oof_rarity_percentile', 'median'),
        median_candidate_repeat_frequency=('candidate_repeat_frequency', 'median'),
    )
)

cell_summary.to_csv(
    REPORT_DIR / 'oof_cell_score_by_repeat.csv', index=False, encoding='utf-8-sig'
)
physical_cell_summary.to_csv(
    REPORT_DIR / 'oof_physical_cell_summary.csv', index=False, encoding='utf-8-sig'
)
component_calibration.to_csv(
    REPORT_DIR / 'oof_fold_calibration.csv', index=False, encoding='utf-8-sig'
)
alpha_search_table.to_csv(
    REPORT_DIR / 'oof_alpha_search.csv', index=False, encoding='utf-8-sig'
)
coverage_sensitivity_audit.to_csv(
    REPORT_DIR / 'oof_coverage_audit.csv', index=False, encoding='utf-8-sig'
)
oof_batch_audit.to_csv(
    REPORT_DIR / 'oof_residual_summary_by_batch.csv', index=False, encoding='utf-8-sig'
)

# Fold artifacts stay on disk; only compact cell/calibration tables were kept.
del oof_residual_result

print('OOF residual cells evaluated:', len(physical_cell_summary))
print('Stable OOF residual candidates:', int(
    physical_cell_summary['is_stable_oof_residual_candidate'].sum()
))
print('Coverage/fold-confounded cells:', int(
    physical_cell_summary['is_residual_confounding'].sum()
))
print('The empirical tail rank is auxiliary evidence, not a fault probability.')
display(oof_batch_audit)
display(physical_cell_summary.head(30))


## 9. Primary retrospective functional degradation analysis (V2)

This analysis separates five different questions instead of compressing them into one anomaly score: absolute-cycle degradation, lifetime-normalized curve shape, sustained SOH landmark lifetime, transient/data quality behavior, and stable rare pattern groups. Shape analysis uses a sustained T90 landmark and never substitutes the arbitrary last observation. Robust FPCA, functional outlyingness, full-curve level/derivative distance, and stable mutual-neighbor groups replace Isolation Forest as the primary method. Rarity scores and selection frequencies are exploratory review evidence, not p-values or fault probabilities. Forecast residual evidence remains an independent auxiliary experiment below.

In [ ]:
from dataclasses import asdict
import json
import sys
import numpy as np
import pandas as pd

functional_module_path = (
    PROJECT_DIR / 'scripts' / 'matr_functional_pattern_analysis.py'
).resolve()
if not functional_module_path.is_file():
    raise FileNotFoundError(
        'The functional analysis module is missing. Keep the complete scripts '
        f'folder with this notebook. Expected file: {functional_module_path}'
    )
try:
    from scripts.matr_functional_pattern_analysis import (
        FunctionalPatternConfig,
        analyze_functional_patterns,
        assert_curve_prediction_parity,
        load_matr_soh_curves,
    )
except ModuleNotFoundError as exc:
    if exc.name in {'scripts', 'scripts.matr_functional_pattern_analysis'}:
        raise ModuleNotFoundError(
            f'Module file exists at {functional_module_path}, but Python is using '
            'a different scripts package. Restart the kernel and run the notebook '
            'from the first cell.'
        ) from exc
    raise

if 'REPORT_DIR' not in globals():
    raise RuntimeError('Run the report-directory setup cell first.')
FIGURE_DIR = REPORT_DIR / 'figures'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

FUNCTIONAL_PATTERN_CONFIG = FunctionalPatternConfig(
    smooth_half_window_cycles=20.0,
    shape_landmark_soh=0.90,
    shape_smooth_half_window_fraction=0.05,
    shape_max_internal_gap_cycles=25.0,
    shape_max_internal_gap_fraction=0.15,
    shape_grid_points=64,
    absolute_grid_points=64,
    absolute_min_coverage=0.75,
    fpca_variance_fraction=0.90,
    fpca_max_components=6,
    level_distance_weight=0.60,
    derivative_distance_weight=0.40,
    rarity_cutoff=3.5,
    individual_method_consensus=2,
    required_selection_frequency=0.80,
    stability_repeats=30,
    group_edge_stability=0.90,
    group_min_size=3,
    group_max_size=5,
    group_max_fraction=0.12,
    group_separation_ratio=2.00,
    group_center_rarity_cutoff=4.00,
    group_min_edge_density=0.67,
    group_min_member_degree=2,
    lifetime_group_min_common_landmarks=3,
    lifetime_group_min_size=2,
    lifetime_group_edge_stability=0.90,
    lifetime_group_separation_ratio=2.00,
    lifetime_group_center_rarity_cutoff=4.00,
    lifetime_group_acceleration_ratio=0.80,
    lifetime_group_pair_duration_ratio=1.25,
    random_state=20260712,
    verbose=True,
)

with open(OOF_OUTPUT_DIR / 'dataset_manifest.json', 'r', encoding='utf-8') as handle:
    functional_dataset_manifest = json.load(handle)
expected_functional_ids = [
    str(item['battery_id'])
    for item in functional_dataset_manifest['batteries']
    if item.get('status') == 'used'
]
print('Loading trusted MATR SOH curves one file at a time...')
all_raw_curves, functional_curve_load_audit = load_matr_soh_curves(
    MATR_DIR,
    expected_battery_count=int(functional_dataset_manifest['n_usable_batteries']),
    expected_battery_ids=expected_functional_ids,
)
print('Loaded physical curves:', all_raw_curves['battery_id'].nunique())
print('Compact SOH rows:', len(all_raw_curves))

functional_prediction_rows = (
    oof_target_parity_rows
    .drop_duplicates(['battery_id', 'target_cycle', 'actual_soh'])
    .reset_index(drop=True)
)
functional_target_parity = assert_curve_prediction_parity(
    all_raw_curves,
    functional_prediction_rows,
    tolerance=1e-6,
)
functional_target_parity_audit = pd.DataFrame([
    {
        'matched_target_rows': int(len(functional_target_parity)),
        'matched_batteries': int(functional_target_parity['battery_id'].nunique()),
        'max_abs_diff': float(functional_target_parity['abs_diff'].max()),
        'target_definition': 'max_discharge_capacity_Ah_over_reference_capacity',
    }
])

print('Running V2 functional degradation analysis...')
functional_pattern_result = analyze_functional_patterns(
    all_raw_curves,
    config=FUNCTIONAL_PATTERN_CONFIG,
)
functional_pattern_summary = functional_pattern_result.pattern_summary.copy()
functional_pattern_summary['analysis_version'] = 'functional_curve_v2'

required_v2_columns = {
    'battery_id', 'cell_id', 'batch_id',
    'shape_analysis_eligible', 'absolute_analysis_eligible',
    'lifetime_analysis_eligible', 'lifetime_ineligibility_reason',
    't90_censoring', 't95_censoring', 't80_censoring',
    'shape_score', 'shape_selection_frequency', 'is_shape_candidate',
    'absolute_pattern_score', 'absolute_selection_frequency',
    'is_absolute_pattern_candidate',
    'lifetime_score', 'lifetime_selection_frequency',
    'is_lifetime_candidate',
    'transient_score', 'transient_selection_frequency',
    'is_transient_candidate',
    'rare_group_id', 'rare_group_stability',
    'is_rare_group_candidate',
    'shape_rare_group_id', 'absolute_rare_group_id',
    'lifetime_rare_group_id',
    'is_shape_rare_group_candidate',
    'is_absolute_rare_group_candidate',
    'is_lifetime_rare_group_candidate',
    'nearest_shape_peers', 'nearest_lifetime_peers', 'review_status',
    'shape_score_distance', 'shape_orthogonal_distance',
    'shape_sd_ratio', 'shape_od_ratio',
}
missing_v2_columns = required_v2_columns - set(functional_pattern_summary.columns)
if missing_v2_columns:
    raise RuntimeError(
        f'V2 functional summary schema mismatch: {sorted(missing_v2_columns)}'
    )

functional_pattern_summary['is_persistent_morphology_candidate'] = (
    functional_pattern_summary[[
        'is_shape_candidate',
        'is_absolute_pattern_candidate',
        'is_shape_rare_group_candidate',
        'is_absolute_rare_group_candidate',
    ]].any(axis=1)
)
functional_pattern_summary['is_lifetime_review_candidate'] = (
    functional_pattern_summary[[
        'is_lifetime_candidate',
        'is_lifetime_rare_group_candidate',
    ]].any(axis=1)
)
functional_pattern_summary['is_persistent_review_candidate'] = (
    functional_pattern_summary['is_persistent_morphology_candidate']
    | functional_pattern_summary['is_lifetime_review_candidate']
)
functional_pattern_summary['is_transient_only_candidate'] = (
    functional_pattern_summary['is_transient_candidate']
    & ~functional_pattern_summary['is_persistent_review_candidate']
)
functional_pattern_summary['evidence_signature'] = functional_pattern_summary.apply(
    lambda row: ''.join([
        'S' if row['is_shape_candidate'] else '-',
        'A' if row['is_absolute_pattern_candidate'] else '-',
        'L' if row['is_lifetime_review_candidate'] else '-',
        'T' if row['is_transient_candidate'] else '-',
        'G' if row['is_rare_group_candidate'] else '-',
    ]),
    axis=1,
)
def assign_v2_review_priority(row):
    if row['is_shape_candidate'] and row['is_absolute_pattern_candidate']:
        return 'P1_shape_and_absolute'
    if (
        row['is_persistent_morphology_candidate']
        and (row['is_lifetime_review_candidate'] or row['is_rare_group_candidate'])
    ) or row['is_lifetime_rare_group_candidate']:
        return 'P2_cross_channel_persistent'
    if row['is_persistent_morphology_candidate']:
        return 'P3_persistent_morphology'
    if row['is_lifetime_review_candidate']:
        return 'P4_lifetime_or_speed'
    if row['is_transient_candidate']:
        return 'QC_transient_only'
    return 'no_stable_review_evidence'
functional_pattern_summary['review_priority'] = functional_pattern_summary.apply(
    assign_v2_review_priority, axis=1
)

forecast_columns = [
    'battery_id', 'expected_oof_repeats', 'evaluated_oof_repeats',
    'has_complete_oof_coverage', 'has_complete_oof_residual_evidence',
    'median_oof_cell_nonconformity_score',
    'median_oof_empirical_tail_p', 'median_oof_rarity_percentile',
    'candidate_repeat_frequency', 'strong_repeat_frequency',
    'is_stable_oof_residual_candidate',
    'is_stable_oof_residual_strong_candidate',
    'is_residual_confounding', 'residual_confounding_type', 'residual_status',
]
if 'physical_cell_summary' in globals():
    available_forecast_columns = [
        column for column in forecast_columns
        if column in physical_cell_summary.columns
    ]
    forecast_evidence = physical_cell_summary[available_forecast_columns].copy()
    forecast_evidence = forecast_evidence.rename(columns={
        column: f'forecast_{column}'
        for column in available_forecast_columns
        if column != 'battery_id'
    })
    functional_pattern_summary = functional_pattern_summary.merge(
        forecast_evidence,
        on='battery_id',
        how='left',
        validate='one_to_one',
    )
    functional_pattern_summary['is_oof_residual_evaluated'] = (
        functional_pattern_summary.get(
            'forecast_has_complete_oof_residual_evidence',
            pd.Series(False, index=functional_pattern_summary.index),
        ).fillna(False).astype(bool)
    )
else:
    functional_pattern_summary['is_oof_residual_evaluated'] = False
if 'forecast_median_oof_empirical_tail_p' not in functional_pattern_summary:
    functional_pattern_summary['forecast_median_oof_empirical_tail_p'] = np.nan

functional_pattern_summary = functional_pattern_summary.sort_values(
    [
        'is_rare_group_candidate',
        'is_shape_candidate',
        'is_absolute_pattern_candidate',
        'is_lifetime_candidate',
        'is_transient_candidate',
        'rare_group_stability',
        'shape_selection_frequency',
    ],
    ascending=[False, False, False, False, False, False, False],
).reset_index(drop=True)

functional_pattern_batch_audit = (
    functional_pattern_summary.groupby('batch_id', as_index=False)
    .agg(
        physical_cells=('battery_id', 'nunique'),
        shape_eligible=('shape_analysis_eligible', 'sum'),
        absolute_eligible=('absolute_analysis_eligible', 'sum'),
        shape_candidates=('is_shape_candidate', 'sum'),
        absolute_pattern_candidates=('is_absolute_pattern_candidate', 'sum'),
        lifetime_candidates=('is_lifetime_candidate', 'sum'),
        transient_candidates=('is_transient_candidate', 'sum'),
        shape_rare_group_members=('is_shape_rare_group_candidate', 'sum'),
        absolute_rare_group_members=('is_absolute_rare_group_candidate', 'sum'),
        lifetime_rare_group_members=('is_lifetime_rare_group_candidate', 'sum'),
        any_rare_group_members=('is_rare_group_candidate', 'sum'),
        persistent_morphology_candidates=(
            'is_persistent_morphology_candidate', 'sum'
        ),
        lifetime_review_candidates=('is_lifetime_review_candidate', 'sum'),
        persistent_review_candidates=('is_persistent_review_candidate', 'sum'),
    )
)

functional_curve_load_audit.to_csv(
    REPORT_DIR / 'functional_curve_load_audit_v2.csv', index=False, encoding='utf-8-sig'
)
functional_target_parity_audit.to_csv(
    REPORT_DIR / 'functional_target_parity_audit_v2.csv', index=False, encoding='utf-8-sig'
)
functional_pattern_result.processed_curves.to_csv(
    REPORT_DIR / 'functional_processed_curves_v2.csv', index=False, encoding='utf-8-sig'
)
functional_pattern_result.shape_representation.to_csv(
    REPORT_DIR / 'functional_shape_representation_v2.csv', index=False, encoding='utf-8-sig'
)
functional_pattern_result.absolute_representation.to_csv(
    REPORT_DIR / 'functional_absolute_representation_v2.csv', index=False, encoding='utf-8-sig'
)
functional_pattern_result.landmark_table.to_csv(
    REPORT_DIR / 'functional_landmarks_v2.csv', index=False, encoding='utf-8-sig'
)
functional_pattern_result.rare_group_summary.to_csv(
    REPORT_DIR / 'functional_rare_groups_v2.csv', index=False, encoding='utf-8-sig'
)
functional_pattern_result.stability_audit.to_csv(
    REPORT_DIR / 'functional_stability_audit_v2.csv', index=False, encoding='utf-8-sig'
)
functional_pattern_summary.to_csv(
    REPORT_DIR / 'functional_pattern_summary_v2.csv', index=False, encoding='utf-8-sig'
)
functional_pattern_batch_audit.to_csv(
    REPORT_DIR / 'functional_summary_by_batch_v2.csv', index=False, encoding='utf-8-sig'
)
with open(REPORT_DIR / 'functional_pattern_config_v2.json', 'w', encoding='utf-8') as handle:
    json.dump(
        {
            'analysis_version': 'functional_curve_v2',
            'config': asdict(FUNCTIONAL_PATTERN_CONFIG),
            'project_dir': str(PROJECT_DIR),
            'matr_dir': str(MATR_DIR),
            'python_executable': str(sys.executable),
            'primary_methods': [
                'robust_fpca_score_and_orthogonal_distance',
                'functional_magnitude_and_shape_outlyingness',
                'level_derivative_curve_distance',
                'stable_rare_mutual_neighbor_groups',
            ],
            'scope': 'retrospective_full_observed_curve',
            'rarity_note': 'not_p_value_not_fault_probability',
        },
        handle,
        ensure_ascii=False,
        indent=2,
    )

print('V2 persistent shape candidates:', int(functional_pattern_summary['is_shape_candidate'].sum()))
print('V2 absolute-cycle candidates:', int(functional_pattern_summary['is_absolute_pattern_candidate'].sum()))
print('V2 lifetime/speed candidates:', int(functional_pattern_summary['is_lifetime_candidate'].sum()))
print('V2 transient/data-quality candidates:', int(functional_pattern_summary['is_transient_candidate'].sum()))
print('V2 shape rare-group members:', int(functional_pattern_summary['is_shape_rare_group_candidate'].sum()))
print('V2 absolute rare-group members:', int(functional_pattern_summary['is_absolute_rare_group_candidate'].sum()))
print('V2 lifetime rare-group members:', int(functional_pattern_summary['is_lifetime_rare_group_candidate'].sum()))
display(functional_pattern_batch_audit)
functional_display_columns = [
    'battery_id', 'batch_id', 'evidence_signature',
    'review_priority', 'review_status',
    'shape_analysis_eligible', 'shape_score',
    'shape_selection_frequency', 'is_shape_candidate',
    'absolute_pattern_score', 'absolute_selection_frequency',
    'is_absolute_pattern_candidate',
    'lifetime_score', 'lifetime_selection_frequency',
    'is_lifetime_candidate',
    'transient_score', 'transient_selection_frequency',
    'is_transient_candidate',
    'shape_rare_group_id', 'absolute_rare_group_id',
    'lifetime_rare_group_id', 'rare_group_stability',
    'nearest_shape_peers', 'nearest_lifetime_peers',
    'is_oof_residual_evaluated',
    'forecast_median_oof_empirical_tail_p',
]
display(functional_pattern_summary[functional_display_columns].head(40))

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

if not functional_pattern_summary['analysis_version'].eq('functional_curve_v2').all():
    raise RuntimeError('Stale or non-V2 functional summary is loaded')

batch_order = ['b1', 'b2', 'b3', 'b4']
batch_colors = {
    'b1': '#2563EB', 'b2': '#F97316',
    'b3': '#16A34A', 'b4': '#7C3AED',
}

def plot_reference_ribbon(axis, representation, candidate_ids, *, title, x_label, y_label):
    if representation.empty:
        axis.text(0.5, 0.5, 'No eligible curves', ha='center', va='center')
        axis.set_title(title)
        return
    pivot = representation.pivot_table(
        index='battery_id', columns='coordinate',
        values='level_value', aggfunc='first',
    ).sort_index(axis=1)
    x = pivot.columns.to_numpy(dtype=float)
    values = pivot.to_numpy(dtype=float)
    axis.fill_between(
        x, np.quantile(values, 0.10, axis=0),
        np.quantile(values, 0.90, axis=0),
        color='#CBD5E1', alpha=0.45, label='batch 10-90% ribbon',
    )
    axis.plot(
        x, np.median(values, axis=0),
        color='#475569', linewidth=2.2, label='batch median',
    )
    for battery_id in sorted(set(candidate_ids) & set(pivot.index.astype(str))):
        axis.plot(
            x, pivot.loc[battery_id].to_numpy(dtype=float),
            linewidth=2.0, label=battery_id,
        )
    axis.set_title(title)
    axis.set_xlabel(x_label)
    axis.set_ylabel(y_label)
    axis.grid(alpha=0.25)
    if candidate_ids:
        axis.legend(fontsize=7)

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True)
for axis, batch_id in zip(axes.ravel(), batch_order):
    part = functional_pattern_result.absolute_representation[
        functional_pattern_result.absolute_representation['batch_id'] == batch_id
    ]
    candidate_ids = set(
        functional_pattern_summary.loc[
            (functional_pattern_summary['batch_id'] == batch_id)
            & functional_pattern_summary['is_absolute_pattern_candidate'],
            'battery_id',
        ].astype(str)
    )
    eligible_n = int(functional_pattern_summary.loc[
        functional_pattern_summary['batch_id'] == batch_id,
        'absolute_analysis_eligible',
    ].sum())
    total_n = int((functional_pattern_summary['batch_id'] == batch_id).sum())
    plot_reference_ribbon(
        axis, part, candidate_ids,
        title=f'{batch_id}: absolute-cycle view ({eligible_n}/{total_n} eligible)',
        x_label='Absolute cycle', y_label='Robust persistent SOH',
    )
plt.tight_layout()
plt.savefig(FIGURE_DIR / '04_v2_absolute_cycle_batch_ribbons.png', dpi=220)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
for axis, batch_id in zip(axes.ravel(), batch_order):
    part = functional_pattern_result.shape_representation[
        functional_pattern_result.shape_representation['batch_id'] == batch_id
    ]
    candidate_ids = set(
        functional_pattern_summary.loc[
            (functional_pattern_summary['batch_id'] == batch_id)
            & functional_pattern_summary['is_shape_candidate'],
            'battery_id',
        ].astype(str)
    )
    eligible_n = int(functional_pattern_summary.loc[
        functional_pattern_summary['batch_id'] == batch_id,
        'shape_analysis_eligible',
    ].sum())
    total_n = int((functional_pattern_summary['batch_id'] == batch_id).sum())
    plot_reference_ribbon(
        axis, part, candidate_ids,
        title=f'{batch_id}: sustained-T90 normalized shape ({eligible_n}/{total_n})',
        x_label='Fraction of sustained-T90 time',
        y_label='Normalized cumulative SOH drop',
    )
plt.tight_layout()
plt.savefig(FIGURE_DIR / '05_v2_landmark_normalized_shapes.png', dpi=220)
plt.show()

shape_fpca = functional_pattern_summary[
    functional_pattern_summary['shape_analysis_eligible']
    & functional_pattern_summary['shape_sd_ratio'].notna()
    & functional_pattern_summary['shape_od_ratio'].notna()
].copy()
fig, axis = plt.subplots(figsize=(9, 7))
for batch_id, group in shape_fpca.groupby('batch_id'):
    axis.scatter(
        group['shape_sd_ratio'], group['shape_od_ratio'],
        color=batch_colors.get(batch_id, '#64748B'),
        alpha=0.65, s=38, label=batch_id,
    )
for _, row in shape_fpca[
    shape_fpca['is_shape_candidate'] | shape_fpca['is_shape_rare_group_candidate']
].iterrows():
    axis.scatter(
        row['shape_sd_ratio'], row['shape_od_ratio'],
        marker='*' if row['is_shape_candidate'] else 'D',
        s=150, facecolors='none', edgecolors='#DC2626', linewidths=1.7,
    )
    axis.annotate(
        row['battery_id'], (row['shape_sd_ratio'], row['shape_od_ratio']),
        xytext=(4, 4), textcoords='offset points', fontsize=8,
    )
axis.axvline(1.0, color='#64748B', linestyle='--', linewidth=1.1)
axis.axhline(1.0, color='#64748B', linestyle='--', linewidth=1.1)
axis.set_xlabel('FPCA score distance / robust cutoff')
axis.set_ylabel('FPCA orthogonal distance / robust cutoff')
axis.set_title('V2 functional outlier map: common variation vs new curve shape')
axis.grid(alpha=0.25)
axis.legend(title='Batch')
plt.tight_layout()
plt.savefig(FIGURE_DIR / '06_v2_fpca_score_vs_orthogonal_distance.png', dpi=220)
plt.show()

channel_flags = [
    'is_shape_candidate', 'is_absolute_pattern_candidate',
    'is_lifetime_candidate', 'is_transient_candidate',
    'is_rare_group_candidate',
]
heatmap_rows = functional_pattern_summary[
    functional_pattern_summary[channel_flags].any(axis=1)
].copy()
fig_height = max(3.5, 0.42 * max(len(heatmap_rows), 1) + 1.8)
fig, axis = plt.subplots(figsize=(10, fig_height))
if heatmap_rows.empty:
    axis.text(0.5, 0.5, 'No stable V2 review candidate', ha='center', va='center', fontsize=13)
    axis.set_axis_off()
else:
    heatmap_rows = heatmap_rows.sort_values(channel_flags, ascending=False)
    evidence_columns = [
        'shape_selection_frequency', 'absolute_selection_frequency',
        'lifetime_selection_frequency', 'transient_selection_frequency',
        'rare_group_stability',
    ]
    evidence_labels = [
        'Shape stability', 'Absolute-cycle stability',
        'Lifetime stability', 'Transient stability', 'Rare-group stability',
    ]
    # pandas 3 Copy-on-Write may return a read-only NumPy view.
    # The following two lines intentionally mask ineligible cells, so request a copy.
    evidence = heatmap_rows[evidence_columns].to_numpy(dtype=float, copy=True)
    evidence[~heatmap_rows['shape_analysis_eligible'].to_numpy(dtype=bool), 0] = np.nan
    evidence[~heatmap_rows['absolute_analysis_eligible'].to_numpy(dtype=bool), 1] = np.nan
    masked = np.ma.masked_invalid(evidence)
    cmap = plt.cm.YlOrRd.copy()
    cmap.set_bad('#E2E8F0')
    image_handle = axis.imshow(masked, aspect='auto', vmin=0.0, vmax=1.0, cmap=cmap)
    axis.set_xticks(np.arange(len(evidence_labels)), evidence_labels, rotation=25, ha='right')
    axis.set_yticks(np.arange(len(heatmap_rows)), heatmap_rows['battery_id'].astype(str))
    for row_index in range(len(heatmap_rows)):
        for column_index in range(len(evidence_labels)):
            value = evidence[row_index, column_index]
            text_value = 'NA' if not np.isfinite(value) else f'{value:.2f}'
            axis.text(column_index, row_index, text_value, ha='center', va='center', fontsize=8)
    axis.set_title('V2 channel selection stability (not fault probability)')
    fig.colorbar(image_handle, ax=axis, label='Selection stability')
plt.tight_layout()
plt.savefig(FIGURE_DIR / '07_v2_channel_evidence_heatmap.png', dpi=220)
plt.show()

candidate_group_ids = set()
for value in functional_pattern_summary.loc[
    functional_pattern_summary['is_rare_group_candidate'], 'rare_group_id'
].astype(str):
    candidate_group_ids.update(group_id for group_id in value.split(';') if group_id)
rare_groups_to_plot = functional_pattern_result.rare_group_summary[
    functional_pattern_result.rare_group_summary['rare_group_id'].isin(candidate_group_ids)
].copy()
if rare_groups_to_plot.empty:
    fig, axis = plt.subplots(figsize=(9, 4))
    axis.text(0.5, 0.5, 'No stable rare degradation-pattern group', ha='center', va='center')
    axis.set_axis_off()
else:
    n_groups = len(rare_groups_to_plot)
    n_columns = 2
    n_rows = int(math.ceil(n_groups / n_columns))
    fig, axes = plt.subplots(n_rows, n_columns, figsize=(14, 5 * n_rows), squeeze=False)
    for axis, (_, group_row) in zip(axes.ravel(), rare_groups_to_plot.iterrows()):
        members = set(str(group_row['member_battery_ids']).split(','))
        group_title = (
            f"{group_row['rare_group_id']} | n={int(group_row['group_size'])} "
            f"| stability={float(group_row['rare_group_stability']):.2f}"
        )
        if group_row['view'] == 'lifetime':
            landmark_labels = ['t95', 't90', 't85', 't80']
            landmark_names = ['T95', 'T90', 'T85', 'T80']
            batch_landmarks = functional_pattern_result.landmark_table[
                functional_pattern_result.landmark_table['batch_id']
                == group_row['batch_id']
            ].copy()
            batch_medians = {}
            for label in landmark_labels:
                observed = batch_landmarks[
                    batch_landmarks[f'{label}_censoring'].eq('observed')
                    & batch_landmarks[label].notna()
                ]
                batch_medians[label] = np.median(
                    observed[label] - observed['cycle_start']
                )
            for battery_id in sorted(members):
                row = batch_landmarks[
                    batch_landmarks['battery_id'] == battery_id
                ].iloc[0]
                ratios = [
                    (row[label] - row['cycle_start']) / batch_medians[label]
                    if row[f'{label}_censoring'] == 'observed' else np.nan
                    for label in landmark_labels
                ]
                axis.plot(landmark_names, ratios, marker='o', linewidth=2.0, label=battery_id)
            axis.axhline(1.0, color='#475569', linestyle='--', label='batch median timing')
            axis.set_title(group_title)
            axis.set_xlabel('Sustained SOH landmark')
            axis.set_ylabel('Landmark duration / batch median')
            axis.grid(alpha=0.25)
            axis.legend(fontsize=8)
        else:
            representation = (
                functional_pattern_result.shape_representation
                if group_row['view'] == 'shape'
                else functional_pattern_result.absolute_representation
            )
            batch_part = representation[
                representation['batch_id'] == group_row['batch_id']
            ]
            plot_reference_ribbon(
                axis, batch_part, members,
                title=group_title,
                x_label=(
                    'Fraction of sustained-T90 time'
                    if group_row['view'] == 'shape' else 'Absolute cycle'
                ),
                y_label=(
                    'Normalized cumulative SOH drop'
                    if group_row['view'] == 'shape' else 'Robust persistent SOH'
                ),
            )
    for axis in axes.ravel()[n_groups:]:
        axis.set_axis_off()
plt.tight_layout()
plt.savefig(FIGURE_DIR / '08_v2_rare_group_curves.png', dpi=220)
plt.show()

## 10. 함수 기반 SOH 곡선 분석과 반복 OOF 예측 잔차 비교

두 결과는 같은 SOH에서 파생된 상호보완적 관점이며 서로 독립적인 정답 검증은 아니다. 배치 내 후보 중첩과 연속 순위 연관성을 함께 보고, coverage/fold 교란 또는 잔차 근거 부족은 별도 상태로 남긴다. 함수 기반 지속 열화 패턴이 주 분석이고, OOF 잔차는 실제 열화가 모델 예측보다 빠른지를 보는 보조 근거다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

'''
# =========================
# Load analysis results
# =========================
REPORT_DIR = OUTPUT_DIR / 'report_style_anomaly_analysis'
FIGURE_DIR = REPORT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print('AUXILIARY CHECK: forecast-residual mismatch; this does not define the primary curve-pattern candidate.')

cycle_path = REPORT_DIR / 'cycle_level_degradation_anomaly_scores.csv'
cell_path = REPORT_DIR / 'cell_level_conformal_summary_by_seed.csv'
physical_path = REPORT_DIR / 'physical_cell_conformal_summary.csv'
alpha_path = REPORT_DIR / 'selected_anomaly_alpha_by_seed.csv'

df = pd.read_csv(cycle_path)
cell_summary = pd.read_csv(cell_path)
physical_cell_summary = pd.read_csv(physical_path)
alpha_by_seed = pd.read_csv(alpha_path)

# =========================
# Candidate selection
# =========================
cand = physical_cell_summary[
    physical_cell_summary['is_physical_strong_candidate'] == True
].copy()
candidate_view = 'strong empirical-tail physical-cell candidates'
if cand.empty:
    cand = physical_cell_summary[
        physical_cell_summary['is_physical_candidate'] == True
    ].copy()
    candidate_view = 'empirical-tail physical-cell candidates'
if cand.empty:
    cand = physical_cell_summary.copy()
    candidate_view = 'exploratory rarity ranking; no p-cutoff candidate'

required_conformal_cols = {
    'median_h10_relative_severity', 'median_h50_relative_severity',
    'median_h100_relative_severity', 'median_cell_nonconformity_score',
    'worst_seed_empirical_p_value', 'is_physical_candidate',
    'is_physical_strong_candidate', 'physical_cell_status',
    'is_repeated_seed_candidate', 'is_repeated_seed_strong_candidate',
    'is_single_seed_candidate', 'is_single_seed_strong_candidate',
}
missing_conformal_cols = required_conformal_cols - set(cand.columns)
if missing_conformal_cols:
    raise RuntimeError(f'Missing conformal columns: {sorted(missing_conformal_cols)}')

cand = cand.sort_values(
    ['is_repeated_seed_strong_candidate', 'is_repeated_seed_candidate',
     'is_single_seed_strong_candidate', 'is_single_seed_candidate',
     'worst_seed_empirical_p_value', 'median_cell_nonconformity_score'],
    ascending=[False, False, False, False, True, False],
).head(12).reset_index(drop=True)
cand['label'] = (
    cand['battery_id'].astype(str)
    + '\nseeds=' + cand['test_seeds'].astype(str)
    + '\n' + cand['physical_cell_status'].astype(str)
)
cand['status_color'] = cand['physical_cell_status'].map({
    'repeated_seed_strong_empirical_tail_candidate': '#991B1B',
    'repeated_seed_empirical_tail_candidate': '#DC2626',
    'single_seed_strong_empirical_tail_candidate': '#F97316',
    'single_seed_empirical_tail_candidate': '#F59E0B',
}).fillna('#64748B')

display(cand)

# =========================
# Plot
# =========================
fig, axes = plt.subplots(1, 3, figsize=(22, 5.5))

x = np.arange(len(cand))
width = 0.25
for offset, horizon, color in [
    (-width, 10, '#60A5FA'), (0.0, 50, '#8B5CF6'), (width, 100, '#EF4444')
]:
    axes[0].bar(
        x + offset, cand[f'median_h{horizon}_relative_severity'],
        width=width, label=f'H{horizon}', color=color, alpha=0.85
    )
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_xticks(x, cand['label'], rotation=50)
axes[0].set_title('Auxiliary Forecast Residual by Horizon')
axes[0].set_ylabel('Alpha-development relative severity')
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend()

axes[1].bar(
    cand['label'], cand['median_cell_nonconformity_score'],
    color=cand['status_color']
)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Auxiliary Forecast-Mismatch Score')
axes[1].set_ylabel('Median of H10/H50/H100 relative means')
axes[1].tick_params(axis='x', rotation=50)
axes[1].grid(axis='y', alpha=0.3)

plot_p = cand['worst_seed_empirical_p_value'].clip(lower=1e-12)
plot_evidence = -np.log10(plot_p)
axes[2].bar(cand['label'], plot_evidence, color=cand['status_color'])
axes[2].axhline(
    -np.log10(ANOMALY_CONFIG.conformal_candidate_p),
    color='#F97316', linestyle='--', linewidth=1.5,
    label=f'candidate p≤{ANOMALY_CONFIG.conformal_candidate_p:.2f}'
)
axes[2].axhline(
    -np.log10(ANOMALY_CONFIG.conformal_strong_p),
    color='#DC2626', linestyle=':', linewidth=1.8,
    label=f'strong p≤{ANOMALY_CONFIG.conformal_strong_p:.2f}'
)
axes[2].set_title('Auxiliary Forecast-Mismatch Tail Evidence')
axes[2].set_ylabel('-log10(worst-seed p-value)')
axes[2].tick_params(axis='x', rotation=50)
axes[2].grid(axis='y', alpha=0.3)
axes[2].legend()
for i, p_value in enumerate(plot_p):
    axes[2].text(i, plot_evidence.iloc[i] + 0.03, f'p={p_value:.3f}', ha='center', fontsize=8)

alpha_label = ', '.join(
    f'seed{int(row.seed)} alpha={row.alpha:.2f}'
    for row in alpha_by_seed.itertuples(index=False)
)
fig.suptitle(
    f'Auxiliary forecast evidence: {candidate_view} ({alpha_label})',
    fontsize=16,
)
plt.tight_layout()

save_path = FIGURE_DIR / 'auxiliary_forecast_mismatch_candidates.png'
plt.savefig(save_path, dpi=220)
plt.show()

print('saved:', save_path)
'''

# Direct cell-by-cell comparison of the two SOH-derived views.
functional_oof_comparison_result = compare_functional_and_oof_evidence(
    functional_pattern_summary,
    physical_cell_summary,
    expected_battery_count=int(functional_dataset_manifest['n_usable_batteries']),
    permutation_repeats=2000,
    random_state=OOF_RESIDUAL_CONFIG.random_state,
)
functional_pattern_summary = (
    functional_oof_comparison_result.cell_comparison.copy()
)
comparison_batch_rows = []
for batch_id, group in functional_pattern_summary.groupby('batch_id', sort=True):
    status_text = group['combined_evidence_status'].astype(str)
    unavailable = status_text.str.contains('unavailable', regex=False)
    confounded = status_text.str.contains('confounded', regex=False)
    eligible = ~(unavailable | confounded)
    f = group['is_persistent_review_candidate'].astype(bool)
    r = group['is_stable_oof_residual_candidate'].astype(bool)
    comparison_batch_rows.append({
        'batch_id': str(batch_id),
        'physical_cells': int(len(group)),
        'eligible_cells': int(eligible.sum()),
        'both': int((eligible & f & r).sum()),
        'functional_only': int((eligible & f & ~r).sum()),
        'residual_only': int((eligible & ~f & r).sum()),
        'neither': int((eligible & ~f & ~r).sum()),
        'confounded': int(confounded.sum()),
        'unavailable': int(unavailable.sum()),
        'agreement_rate_among_eligible': float((f[eligible] == r[eligible]).mean())
        if eligible.any() else np.nan,
    })
functional_oof_comparison_by_batch = pd.DataFrame(comparison_batch_rows)
functional_oof_overlap_audit = (
    functional_oof_comparison_result.overlap_audit.copy()
)
functional_oof_rank_agreement_audit = (
    functional_oof_comparison_result.rank_agreement_audit.copy()
)

functional_pattern_summary.to_csv(
    REPORT_DIR / 'functional_vs_oof_cell_comparison.csv',
    index=False, encoding='utf-8-sig',
)
functional_oof_comparison_by_batch.to_csv(
    REPORT_DIR / 'functional_vs_oof_comparison_by_batch.csv',
    index=False, encoding='utf-8-sig',
)
functional_oof_overlap_audit.to_csv(
    REPORT_DIR / 'functional_vs_oof_overlap_audit.csv',
    index=False, encoding='utf-8-sig',
)
functional_oof_rank_agreement_audit.to_csv(
    REPORT_DIR / 'functional_vs_oof_rank_agreement_audit.csv',
    index=False, encoding='utf-8-sig',
)

status_colors = {
    'concordant_persistent_and_forecast': '#7F1D1D',
    'functional_only_review': '#2563EB',
    'forecast_only_review': '#F97316',
    'transient_qc_only': '#A855F7',
    'comparison_confounded': '#64748B',
    'comparison_unavailable': '#CBD5E1',
    'functional_primary_residual_confounded': '#1D4ED8',
    'functional_primary_residual_unavailable': '#60A5FA',
    'no_joint_evidence': '#94A3B8',
}
fig, axes = plt.subplots(1, 3, figsize=(21, 5.8))
batch_plot = functional_oof_comparison_by_batch.set_index('batch_id')
batch_plot[[
    'both', 'functional_only', 'residual_only', 'neither',
    'confounded', 'unavailable',
]].plot(
    kind='bar', stacked=True, ax=axes[0],
    color=[
        '#7F1D1D', '#2563EB', '#F97316', '#CBD5E1',
        '#64748B', '#E2E8F0',
    ],
)
axes[0].set_title('Method agreement by batch')
axes[0].set_xlabel('Batch')
axes[0].set_ylabel('Eligible physical cells')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.25)

comparison_plot_status = (
    functional_pattern_summary['combined_evidence_status'].astype(str)
)
comparison_plot = functional_pattern_summary[
    ~comparison_plot_status.str.contains('confounded|unavailable', regex=True)
].copy()
for source_column, rank_column in [
    ('shape_score', 'shape_within_batch_percentile'),
    ('absolute_pattern_score', 'absolute_within_batch_percentile'),
]:
    comparison_plot[rank_column] = (
        comparison_plot.groupby('batch_id')[source_column]
        .rank(method='average', pct=True) * 100.0
    )
for status, group in comparison_plot.groupby('combined_evidence_status'):
    color = status_colors.get(status, '#94A3B8')
    axes[1].scatter(
        pd.to_numeric(group['shape_within_batch_percentile'], errors='coerce'),
        pd.to_numeric(group['median_oof_rarity_percentile'], errors='coerce'),
        label=status, color=color, alpha=0.75, s=38,
    )
    axes[2].scatter(
        pd.to_numeric(group['absolute_within_batch_percentile'], errors='coerce'),
        pd.to_numeric(group['median_oof_rarity_percentile'], errors='coerce'),
        label=status, color=color, alpha=0.75, s=38,
    )
for axis, x_column, title in [
    (axes[1], 'shape within-batch percentile', 'Functional shape vs OOF residual rarity'),
    (axes[2], 'absolute within-batch percentile', 'Absolute curve vs OOF residual rarity'),
]:
    axis.set_xlabel(x_column)
    axis.set_ylabel('Median OOF residual rarity percentile')
    axis.set_title(title)
    axis.grid(alpha=0.25)
axes[2].legend(loc='best', fontsize=7)
fig.suptitle(
    'Functional degradation evidence vs repeated battery-level OOF forecast residuals',
    fontsize=14,
)
plt.tight_layout()
comparison_figure_path = (
    FIGURE_DIR / '10_functional_vs_oof_residual_comparison.png'
)
plt.savefig(comparison_figure_path, dpi=220)
plt.show()

comparison_display_columns = [
    'battery_id', 'batch_id', 'combined_review_priority',
    'combined_evidence_status', 'review_priority',
    'residual_status', 'median_oof_rarity_percentile',
    'candidate_repeat_frequency', 'is_persistent_review_candidate',
    'is_stable_oof_residual_candidate', 'is_transient_only_candidate',
]
comparison_display = functional_pattern_summary.sort_values(
    ['combined_review_priority', 'median_oof_rarity_percentile'],
    ascending=[True, False],
)[comparison_display_columns]
print('Agreement is supporting evidence, not validation against a true anomaly label.')
display(functional_oof_overlap_audit)
display(functional_oof_rank_agreement_audit)
display(functional_oof_comparison_by_batch)
display(comparison_display.head(50))
print('saved:', comparison_figure_path)


In [ ]:
# V2 full-observed-range detail plot for persistent review candidates
if 'functional_pattern_summary' not in globals():
    functional_pattern_summary = pd.read_csv(
        REPORT_DIR / 'functional_pattern_summary_v2.csv'
    )
    if not functional_pattern_summary['analysis_version'].eq('functional_curve_v2').all():
        raise RuntimeError('Stale/non-V2 summary loaded')
if 'functional_pattern_result' not in globals():
    raise RuntimeError('Run the V2 primary functional analysis cell first.')

detail_flag_columns = [
    'is_shape_candidate',
    'is_absolute_pattern_candidate',
    'is_lifetime_candidate',
    'is_rare_group_candidate',
    'is_stable_oof_residual_candidate',
]
detail_flag_columns = [
    column for column in detail_flag_columns
    if column in functional_pattern_summary.columns
]
detail_candidate_ids = set(
    functional_pattern_summary.loc[
        functional_pattern_summary[detail_flag_columns].any(axis=1),
        'battery_id',
    ].astype(str)
)
processed_v2 = functional_pattern_result.processed_curves
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True)
for axis, batch_id in zip(axes.ravel(), ['b1', 'b2', 'b3', 'b4']):
    batch_curves = processed_v2[processed_v2['batch_id'] == batch_id]
    batch_candidate_ids = detail_candidate_ids & set(
        batch_curves['battery_id'].astype(str)
    )
    for battery_id, one in batch_curves.groupby('battery_id'):
        one = one.sort_values('cycle')
        if battery_id in batch_candidate_ids:
            continue
        axis.plot(
            one['cycle'],
            one['persistent_trend_soh'],
            color='#CBD5E1',
            linewidth=0.8,
            alpha=0.55,
        )
    for battery_id in sorted(batch_candidate_ids):
        one = batch_curves[
            batch_curves['battery_id'] == battery_id
        ].sort_values('cycle')
        summary_row = functional_pattern_summary.loc[
            functional_pattern_summary['battery_id'] == battery_id
        ].iloc[0]
        status = str(summary_row.get(
            'combined_review_priority', summary_row['review_priority']
        ))
        axis.plot(
            one['cycle'],
            one['persistent_trend_soh'],
            linewidth=2.1,
            label=f'{battery_id} [{status}]',
        )
    axis.axhline(0.90, color='#94A3B8', linestyle='--', linewidth=0.9)
    axis.axhline(0.80, color='#64748B', linestyle=':', linewidth=0.9)
    axis.set_title(
        f'{batch_id}: functional/OOF review cells '
        f'({len(batch_candidate_ids)}/{batch_curves["battery_id"].nunique()})'
    )
    axis.set_xlabel('Cycle')
    axis.set_ylabel('SOH (model target definition)')
    axis.grid(alpha=0.25)
    if batch_candidate_ids:
        axis.legend(fontsize=7)
plt.tight_layout()
save_path = FIGURE_DIR / '11_full_range_functional_oof_review_cells.png'
plt.savefig(save_path, dpi=220)
plt.show()
print('saved:', save_path)
if not detail_candidate_ids:
    print('No functional/OOF review cell; no ranking fallback was forced.')

In [ ]:
# V2 result interpretation and audit tables
print('V2 labels describe stable cohort rarity, not confirmed battery faults.')
print('Shape and lifetime are separate channels; a cell may legitimately have both labels.')
print('Transient-only cells are not presented as persistent degradation-pattern candidates.')
print('Cells without a sustained T90 remain shape-ineligible rather than being called normal.')

v2_channel_totals = pd.DataFrame([
    {
        'physical_cells': int(len(functional_pattern_summary)),
        'shape_eligible': int(functional_pattern_summary['shape_analysis_eligible'].sum()),
        'absolute_eligible': int(functional_pattern_summary['absolute_analysis_eligible'].sum()),
        'shape_candidates': int(functional_pattern_summary['is_shape_candidate'].sum()),
        'absolute_candidates': int(functional_pattern_summary['is_absolute_pattern_candidate'].sum()),
        'lifetime_candidates': int(functional_pattern_summary['is_lifetime_candidate'].sum()),
        'transient_candidates': int(functional_pattern_summary['is_transient_candidate'].sum()),
        'shape_rare_group_members': int(
            functional_pattern_summary['is_shape_rare_group_candidate'].sum()
        ),
        'absolute_rare_group_members': int(
            functional_pattern_summary['is_absolute_rare_group_candidate'].sum()
        ),
        'lifetime_rare_group_members': int(
            functional_pattern_summary['is_lifetime_rare_group_candidate'].sum()
        ),
        'persistent_morphology_candidates': int(
            functional_pattern_summary['is_persistent_morphology_candidate'].sum()
        ),
        'lifetime_review_candidates': int(
            functional_pattern_summary['is_lifetime_review_candidate'].sum()
        ),
        'persistent_review_candidates': int(
            functional_pattern_summary['is_persistent_review_candidate'].sum()
        ),
        'transient_only_candidates': int(
            functional_pattern_summary['is_transient_only_candidate'].sum()
        ),
    }
])
display(v2_channel_totals)
display(functional_pattern_batch_audit)
display(
    functional_pattern_summary.groupby(
        ['review_priority', 'evidence_signature'], as_index=False
    ).agg(physical_cells=('battery_id', 'nunique'))
)
if functional_pattern_result.rare_group_summary.empty:
    print('No descriptive rare group component was produced.')
else:
    display(
        functional_pattern_result.rare_group_summary.sort_values(
            ['is_base_rare_group_candidate', 'rare_group_stability', 'separation_ratio'],
            ascending=[False, False, False],
        )
    )